# Configuration EMR  sur le Cloud (AWS)


# Commencement aws: Création utilisateur


**Création utilisateur et autorisations louées**

![Accès au serveur d'historique spark](image_aws/IAM-utilisateur.png)

**nom utilisateur: cli-lylber**

<u>On fournit des autorisations administratifs pour S3 et en général. Clé activée<u>

![Accès au autorisations de l'utilisateur](image_aws/autorisation_utilisateur.png)

<u> Possibilité de crée des autorisations sur mesure (politique en ligne). Ici permettre la syncronisation entre son compartiment (bucket: lylber-p8-data) et un dossier local<u>

![Accès au autorisations de l'utilisateur](image_aws/autorisation_personnalisee.png)

<u>Télécharger le csv qui contient la paire de clés<u>

# Commencement local: pip install awscli


In [2]:
!where aws

C:\Users\Hilbert\anaconda3\envs\spark_env\Scripts\aws
C:\Users\Hilbert\anaconda3\envs\spark_env\Scripts\aws.cmd
C:\Program Files\Amazon\AWSCLIV2\aws.exe


**Configuration  aws**

![config sur console](image_aws/aws_configure.png)

<b>- 
- On rentre l'ID et la clé secrete pour l'utilisateur cli-lylber  
- On choisit la localisation eu-west-3 pour avoir les serveurs à Paris : rapidité et protection des données selon la legislature européenne mais coût plus élévé
- format de sortie par défault JSON<b>

# Création d'un bucket s3

<b>- 
- commande powershell: aws s3 mb s3://lylber-p8-data
- resultat: make_bucket: lylber-p8-data
<b>

![ls s3](image_aws/aws_mb_s3.png)

<b>On se positionne dans le dossier Test et on fait une synchonisation avec le bucket
- commande powershell: aws s3 sync . s3://lylber-p8-data/Test <b>

![ls s3](image_aws/bucket_ls.png)

<b>On y retrouve les chargements de packages dans  bootstrap-emr.sh <b>

![bootstrap-emr.sh](image_aws/package.png)

<b>Attention il y aura un probléme d'import avec s3fs il faut importer boto3 et s3fs dans  une meme ligne pip<b>

![bootstrap-emr.sh](image_aws/boto_s3fs.png)


<b>-  obtenir le nombre d'images totales et la taille du dossier 
- commande powershell: aws s3 ls --recursive --summarize --human-readable s3://lylber-p8-data/Test/
<b>

![ls s3](image_aws/nrb_taille_test.png)

![ls s3 test](image_aws/s3_test.png)


# Création clés SSH

Les clés SSH sont un moyen sécurisé d'établir une connexion entre deux machines via un réseau non sécurisé 
Elles sont éssentiels pour lancer une machine EMR (qui se repose sur les machines EC2)  
RSA: Recommandé pour une compatibilité maximale  
<b>
.pem: OpenSSH est a choisir car compatible avec linux (mais on est sur windows 11)  
.pkk: PuTTy compatible windows  (celle qu'on va utiliser)

<b>

![clé putty](image_aws/paires_clé_putty.png)


# Autorisation du tunnel SSH dans AWS 



Par défaut, AWS bloque généralement l'accès au port 22 (utilisé par SSH) pour des raisons de sécurité, afin de prévenir les accès non autorisés aux instances EC2. En autorisant le tunnel SSH à travers le port 22 dans le firewall d'AWS, on peut établir une connexion sécurisée et accéder à JupyterHub et aux logs de Spark qui se trouvent dans le réseau local du "driver" dans AWS depuis notre machine locale


![security group](image_aws/goupe_sec.png)


![security group](image_aws/group_sec_p8.png)

<b> Néanmoins lors de la création de notre EMR deux groupes securités se créent: ElasticMapReduce-master et ElasticMapReduce-slave <b>

![security group EMR](image_aws/emr_secu1.png)

<b> on ajoute l'autorisation au port 22 pour les connection ipv4 et ipv6<b>

![security group EMR](image_aws/emr_secu2..png)


#  Configuration du serveur EMR

<b> On selectionne le téléchargement spark, hadoop, tensorflow et jupyter hub. Attention à prendre la meme version que dans le notebook de l'EMR pour compatibilité <b>

![security group EMR](image_aws/config_cluster.png)

<b> pour la configuration de cluster on selectionne Flottes d'instances flexibles.
- on choisit une instance m5.xlarge comme instance maitre
- on choisit une instance m5.xlarge comme instance noeud

![security group EMR](image_aws/config_instance.png)

<b> Ensuite on laisse les paramétres par défaut pour la dimension et mise en service du cluster <b>

<b> selection configuration sécurité EMR <b>

![security group EMR](image_aws/config_cluster_secu.png)


<b> - on selectionne : Résilier automatiquement le cluster après le temps d'inactivité (Recommandé)   <b>
<b> - on ajoute une action d'armorçage: installation de packages à tous niveau du cluster <b>

![image.png](image_aws/amorcage.png)

<b> ajout securité de la paires de clés putty <b>

![image.png](image_aws/config_putty.png)

<b> configuration securité de notre cluster <b>

![image.png](image_aws/config_secu_final.png)

<b> enfin on donne l'autorisation en lecture et écriture sur notre espace s3 à nos instances <b>

![image.png](image_aws/config_autorisation.png)

<b> on peut lancer notre cluster, il n' y plus qu ' à attendre l'initialisation <b>




![image.png](image_aws/demarage.png)
![image.png](image_aws/amorcage_cluster.png)
![image.png](image_aws/attente.png)


# Etablir le tunnel SSH avec le cluster

<b> cliquer sur connexion au noeud primaire à l'aide de SSH, aller dans l'onglet windows et copier le lien 
- ouvrir puTTy et suivre les consignes en ajoutant ceci : <b>

![image.png](image_aws/putty_lien.png)
![image.png](image_aws/putty_config.png)


<b> Mettre sa clé ssh et accepter <b>

![image.png](image_aws/putty_ok.png)
![image.png](image_aws/putty_emr.png)

<b> Permettre à notre navigateur d'utiliser le tunnel SSH <b>

![image.png](image_aws/proxy.png)
![image.png](image_aws/apres_ssh.png)



# Connection jupyter Hub (onglet Application) et lancement du notebook ci-dessous

# calcul


![image.png](image_aws/kernel.png)


In [1]:
#  L'exécution de cette cellule démarre l'application Spark

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
0,application_1708134068010_0001,pyspark,idle,Link,Link,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [4]:
%%info

ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
0,application_1708134068010_0001,pyspark,idle,Link,Link,✔


In [2]:
%pip list


Package                      Version
---------------------------- -----------
aiobotocore                  1.2.2
aiohttp                      3.7.4.post0
aioitertools                 0.7.1
alembic                      1.4.1
argon2-cffi                  20.1.0
async-generator              1.10
async-timeout                3.0.1
attrs                        19.3.0
autovizwidget                0.18.0
backcall                     0.1.0
bleach                       3.1.3
blinker                      1.4
boto3                        1.16.52
botocore                     1.19.52
cachetools                   4.2.1
certifi                      2020.12.5
certipy                      0.1.3
cffi                         1.14.0
chardet                      3.0.4
conda                        4.8.2
conda-package-handling       1.6.0
cryptography                 2.8
cycler                       0.10.0
decorator                    4.4.2
defusedxml                   0.6.0
entrypoints                  0.3


In [6]:
import pandas as pd
import numpy as np
import io
import os
import tensorflow as tf
from PIL import Image
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras import Model
from pyspark.sql.functions import col, pandas_udf, PandasUDFType, element_at, split
from pyspark.sql.functions import col, pandas_udf, PandasUDFType, element_at, split
from pyspark.sql import SparkSession

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [7]:
print(np.__version__)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

1.16.5

In [8]:
print(tf.__version__)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

2.4.1

In [9]:
print(tensorflow.__version__)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

2.4.1

In [10]:
PATH = 's3://lylber-p8-data'
PATH_Data = PATH+'/Test'
PATH_Result = PATH+'/Results'
print('PATH:        '+\
      PATH+'\nPATH_Data:   '+\
      PATH_Data+'\nPATH_Result: '+PATH_Result)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

PATH:        s3://lylber-p8-data
PATH_Data:   s3://lylber-p8-data/Test
PATH_Result: s3://lylber-p8-data/Results

In [11]:
images = spark.read.format("binaryFile") \
  .option("pathGlobFilter", "*.jpg") \
  .option("recursiveFileLookup", "true") \
  .load(PATH_Data)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [12]:
images.show(5)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------+-------------------+------+--------------------+
|                path|   modificationTime|length|             content|
+--------------------+-------------------+------+--------------------+
|s3://lylber-p8-da...|2024-02-13 20:42:44|  7353|[FF D8 FF E0 00 1...|
|s3://lylber-p8-da...|2024-02-13 20:42:45|  7350|[FF D8 FF E0 00 1...|
|s3://lylber-p8-da...|2024-02-13 20:42:45|  7349|[FF D8 FF E0 00 1...|
|s3://lylber-p8-da...|2024-02-13 20:42:45|  7348|[FF D8 FF E0 00 1...|
|s3://lylber-p8-da...|2024-02-13 20:42:45|  7328|[FF D8 FF E0 00 1...|
+--------------------+-------------------+------+--------------------+
only showing top 5 rows

In [13]:
images = images.withColumn('label', element_at(split(images['path'], '/'),-2))
print(images.printSchema())
print(images.select('path','label').show(5,False))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- path: string (nullable = true)
 |-- modificationTime: timestamp (nullable = true)
 |-- length: long (nullable = true)
 |-- content: binary (nullable = true)
 |-- label: string (nullable = true)

None
+-------------------------------------------------+----------+
|path                                             |label     |
+-------------------------------------------------+----------+
|s3://lylber-p8-data/Test/Watermelon/r_106_100.jpg|Watermelon|
|s3://lylber-p8-data/Test/Watermelon/r_109_100.jpg|Watermelon|
|s3://lylber-p8-data/Test/Watermelon/r_108_100.jpg|Watermelon|
|s3://lylber-p8-data/Test/Watermelon/r_107_100.jpg|Watermelon|
|s3://lylber-p8-data/Test/Watermelon/r_95_100.jpg |Watermelon|
+-------------------------------------------------+----------+
only showing top 5 rows

None

In [14]:
model = MobileNetV2(weights='imagenet',
                    include_top=True,
                    input_shape=(224, 224, 3))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

14540800/14536120 [==============================] - 1s 0us/step

In [15]:
new_model = Model(inputs=model.input,
                  outputs=model.layers[-2].output)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [16]:
brodcast_weights = sc.broadcast(new_model.get_weights())

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [17]:
new_model.summary()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Model: "model"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 224, 224, 3) 0                                            
__________________________________________________________________________________________________
Conv1 (Conv2D)                  (None, 112, 112, 32) 864         input_1[0][0]                    
__________________________________________________________________________________________________
bn_Conv1 (BatchNormalization)   (None, 112, 112, 32) 128         Conv1[0][0]                      
__________________________________________________________________________________________________
Conv1_relu (ReLU)               (None, 112, 112, 32) 0           bn_Conv1[0][0]                   
______________________________________________________________________________________________

In [18]:
def model_fn():
    """
    Returns a MobileNetV2 model with top layer removed 
    and broadcasted pretrained weights.
    """
    model = MobileNetV2(weights='imagenet',
                        include_top=True,
                        input_shape=(224, 224, 3))
    for layer in model.layers:
        layer.trainable = False
    new_model = Model(inputs=model.input,
                  outputs=model.layers[-2].output)
    new_model.set_weights(brodcast_weights.value)
    return new_model

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [19]:
def preprocess(content):
    """
    Preprocesses raw image bytes for prediction.
    """
    img = Image.open(io.BytesIO(content)).resize([224, 224])
    arr = img_to_array(img)
    return preprocess_input(arr)

def featurize_series(model, content_series):
    """
    Featurize a pd.Series of raw images using the input model.
    :return: a pd.Series of image features
    """
    input = np.stack(content_series.map(preprocess))
    preds = model.predict(input)
    # For some layers, output features will be multi-dimensional tensors.
    # We flatten the feature tensors to vectors for easier storage in Spark DataFrames.
    output = [p.flatten() for p in preds]
    return pd.Series(output)

@pandas_udf('array<float>', PandasUDFType.SCALAR_ITER)
def featurize_udf(content_series_iter):
    '''
    This method is a Scalar Iterator pandas UDF wrapping our featurization function.
    The decorator specifies that this returns a Spark DataFrame column of type ArrayType(FloatType).

    :param content_series_iter: This argument is an iterator over batches of data, where each batch
                              is a pandas Series of image data.
    '''
    # With Scalar Iterator pandas UDFs, we can load the model once and then re-use it
    # for multiple data batches.  This amortizes the overhead of loading big models.
    model = model_fn()
    for content_series in content_series_iter:
        yield featurize_series(model, content_series)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

/usr/lib/spark/python/lib/pyspark.zip/pyspark/sql/pandas/functions.py:392: UserWarning: In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.

In [20]:
features_df = images.repartition(24).select(col("path"),
                                            col("label"),
                                            featurize_udf("content").alias("features")
                                           )

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [21]:
print(PATH_Result)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

s3://lylber-p8-data/Results

In [22]:
features_df.write.mode("overwrite").parquet(PATH_Result)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…


![image.png](image_aws/spark_app1.png)
![image.png](image_aws/spark_app2.png)
![image.png](image_aws/spark_stage3.png)
c

# PCA

![image.png](image_aws/pca.png)

In [1]:
#  L'exécution de cette cellule démarre l'application Spark

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
5,application_1708202113228_0006,pyspark,idle,Link,Link,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
%info

ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
5,application_1708202113228_0006,pyspark,idle,Link,Link,✔


In [4]:
spark.sparkContext.getConf().get('spark.driver.memory')


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

'1000M'

In [5]:
%%configure -f 
{"driverMemory": "6000M"}

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
6,application_1708202113228_0007,pyspark,idle,Link,Link,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
6,application_1708202113228_0007,pyspark,idle,Link,Link,✔


In [6]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.linalg import Vectors
from pyspark.ml.feature import PCA
from pyspark.sql.functions import udf

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [7]:
df = pd.read_parquet('s3://lylber-p8-data/Results', engine='pyarrow')

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [8]:
df.head()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

                                                path  ...                                           features
0   s3://lylber-p8-data/Test/Watermelon/r_90_100.jpg  ...  [0.12482018, 0.026548253, 0.0, 0.0005341537, 0...
1  s3://lylber-p8-data/Test/Pineapple Mini/33_100...  ...  [0.0, 5.0096755, 0.0, 0.0, 0.0, 0.0, 0.1420599...
2    s3://lylber-p8-data/Test/Watermelon/244_100.jpg  ...  [0.00083662715, 0.015711863, 0.0, 0.0, 1.37703...
3    s3://lylber-p8-data/Test/Watermelon/137_100.jpg  ...  [0.00825152, 0.38440272, 0.04856346, 0.0, 2.44...
4  s3://lylber-p8-data/Test/Cauliflower/r_184_100...  ...  [0.0, 0.6634527, 3.1199622, 0.0, 0.0, 0.0, 0.0...

[5 rows x 3 columns]

In [9]:
df.loc[0,'features'].shape

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

(1280,)

In [10]:
df.shape

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

(22688, 3)

In [11]:
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", "1024")
# Conversion du DataFrame Pandas en DataFrame PySpark
df['features'] = df['features'].apply(lambda x: Vectors.dense(x))
data_spark = spark.createDataFrame(df)

# Création d'un VectorAssembler qui combine une liste de colonnes en une seule colonne de vecteur.
vecAssembler = VectorAssembler(inputCols=["features"], outputCol="features_vec")

# Utilisation du VectorAssembler pour transformer notre DataFrame.
data_spark = vecAssembler.transform(data_spark)

variances = []
total_features = len(data_spark.select("features").first()[0])
resultats = []
print('start -----------------')

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

start -----------------

In [12]:
pca = PCA(k=80, inputCol="features_vec", outputCol="pcaFeatures") # nombre max que je souhaite
pcaModel = pca.fit(data_spark) # fit the data to pca to make the model

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [15]:
import numpy as np
cumValues = pcaModel.explainedVariance.cumsum() # get the cumulative values
list(cumValues).reverse()
K=np.argmax(cumValues >= 0.8)
print(f'parametre k: {K}')
# plot the graph 
# plt.figure(figsize=(10,8))
# plt.plot(range(1,16), cumValues, marker = 'o', linestyle='--')
# plt.title('variance by components')
# plt.xlabel('num of components')
# plt.ylabel('cumulative explained variance')


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

parametre k: 79

In [22]:
cumValues

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

array([0.10140993, 0.18146972, 0.24497274, 0.29512059, 0.33047488,
       0.3596348 , 0.38736694, 0.41021583, 0.43007527, 0.4491668 ,
       0.46570387, 0.48036675, 0.49434859, 0.50803436, 0.52138226,
       0.5338826 , 0.54543343, 0.55618264, 0.56598301, 0.57569308,
       0.58485138, 0.59313437, 0.60102209, 0.60852993, 0.61567809,
       0.62275045, 0.62958686, 0.63580522, 0.64192727, 0.64781775,
       0.6535209 , 0.65909672, 0.66437057, 0.66942059, 0.67420583,
       0.67889421, 0.68348948, 0.68783587, 0.69207105, 0.69616312,
       0.70006549, 0.70395477, 0.70779923, 0.71152293, 0.71515284,
       0.71857504, 0.72197207, 0.72530281, 0.72859337, 0.73183042,
       0.73502977, 0.73808652, 0.7410565 , 0.74393956, 0.74676961,
       0.7495623 , 0.75231607, 0.75500415, 0.75760227, 0.76012722,
       0.76260926, 0.76506099, 0.76741253, 0.76973087, 0.77202279,
       0.77426796, 0.77646267, 0.77863137, 0.7807576 , 0.78287009,
       0.7849326 , 0.78692533, 0.78885697, 0.79076406, 0.79263

In [16]:
# Création d'un objet PCA 
pca = PCA(k=K, inputCol="features_vec", outputCol="pcaFeatures")

# Entraînement du modèle PCA sur les données
model = pca.fit(data_spark)

# Transformation des données avec le modèle PCA
result = model.transform(data_spark)

# Suppression de la colonne 'features_vec'
result = result.drop('features_vec')

# Affichage des résultats
result.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------+--------------+--------------------+--------------------+
|                path|         label|            features|         pcaFeatures|
+--------------------+--------------+--------------------+--------------------+
|s3://lylber-p8-da...|    Watermelon|[0.12482018023729...|[-2.8237825964401...|
|s3://lylber-p8-da...|Pineapple Mini|[0.0,5.0096755027...|[-5.1251192509570...|
|s3://lylber-p8-da...|    Watermelon|[8.36627150420099...|[-1.8078518576448...|
|s3://lylber-p8-da...|    Watermelon|[0.00825151987373...|[-2.4721665769438...|
|s3://lylber-p8-da...|   Cauliflower|[0.0,0.6634526848...|[-4.4352586413862...|
|s3://lylber-p8-da...|   Cauliflower|[0.0,0.2231649905...|[-5.1701454317279...|
|s3://lylber-p8-da...|   Cauliflower|[0.0,0.4306903779...|[-4.4896372468841...|
|s3://lylber-p8-da...|     Raspberry|[0.24571064114570...|[-1.9875168084382...|
|s3://lylber-p8-da...|     Pineapple|[0.0,4.7301945686...|[-5.5753559854939...|
|s3://lylber-p8-da...|     Pineapple|[0.

In [19]:
result.write.mode('overwrite').parquet('s3://lylber-p8-data/PCA')

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [20]:
df= spark.read.parquet('s3://lylber-p8-data/PCA', engine='pyarrow').toPandas()


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [21]:
df

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

                                                    path  ...                                        pcaFeatures
0      s3://lylber-p8-data/Test/Grape White 4/r_23_10...  ...  [6.940769289226938, 3.450862918063562, -10.596...
1       s3://lylber-p8-data/Test/Granadilla/r_14_100.jpg  ...  [4.852544574289753, -1.824743716984738, -5.312...
2        s3://lylber-p8-data/Test/Chestnut/r2_15_100.jpg  ...  [-3.2509751057320275, 7.335834535295284, -5.67...
3      s3://lylber-p8-data/Test/Cactus fruit/r_28_100...  ...  [-2.4739546938979116, 0.9812893500593529, -1.4...
4          s3://lylber-p8-data/Test/Pepino/r_247_100.jpg  ...  [4.659343556714836, -1.4560651054366311, -6.79...
...                                                  ...  ...                                                ...
22683        s3://lylber-p8-data/Test/Salak/r_16_100.jpg  ...  [0.08430093472185668, 3.655702432435665, 2.227...
22684  s3://lylber-p8-data/Test/Pear Monster/r_200_10...  ...  [-1.9489799873868843, -4.12716844

![image.png](image_aws/exit.png)

![image.png](image_aws/s3_pca.png)